# Machine Learning Modeling

In [5]:
# imports
import pandas as pd 
import numpy as np 
import json
import joblib
import warnings
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report,
                             confusion_matrix,
                             accuracy_score,
                             f1_score)
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

print("Imports done ")

Imports done 


# Model A -- Visit Risk Classification
## Business Purpose:
Predict wether a hospital visit represents Low,Medium, or High operational and clinical risk.

In [6]:
# load the dataset
df = pd.read_csv("../outputs/model_table.csv",parse_dates=["registration_date","visit_date","billing_date"])
print("Shape: ",df.shape)
df.head()

Shape:  (25000, 30)


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,...,risk_numeric,claim_numeric,is_rejected,days_since_registration,visit_frequency,avg_los_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag
0,2,15,F,Mumbai,CareOne,0,2025-12-27,8,2026-01-01,General,...,NaN,2,1,5,4,21.120000,0.256876,1,3,False
1,12,3,M,Bangalore,CareOne,0,2025-08-13,65,2026-01-01,ICU,...,2.0,2,1,141,8,23.750000,0.256876,1,3,True
2,129,44,M,Pune,MediCareX,1,2025-07-20,651,2026-01-01,ICU,...,2.0,1,0,165,3,32.460000,0.242556,1,3,True
3,133,47,F,Delhi,CareOne,1,2025-11-02,670,2026-01-01,General,...,NaN,0,0,60,3,30.056667,0.256876,1,3,False
4,139,14,F,Chennai,SecureLife,1,2025-02-05,706,2026-01-01,Cardiology,...,NaN,0,0,330,9,29.030000,0.157496,1,3,True


In [7]:
# risk fea\tures
risk_target = "risk_score"
risk_features = [
    "age",
    "gender",
    "city",
    "insurance_provider",
    "chronic_flag",
    "department",
    "visit_type",
    "doctor_id",
    "length_of_stay_hours",
    "days_since_registration",
    "visit_frequency",
    "avg_los_per_patient",
    "visit_month",
    "visit_dayofweek"
]
risk_df = df[risk_features + [risk_target, "visit_date"]].copy()
risk_df = risk_df.dropna(subset=[risk_target,"visit_date"])
print("Risk dataset shape:", risk_df.shape)
risk_df.head()

Risk dataset shape: (25000, 16)


,age,gender,city,insurance_provider,chronic_flag,department,visit_type,doctor_id,length_of_stay_hours,days_since_registration,visit_frequency,avg_los_per_patient,visit_month,visit_dayofweek,risk_score,visit_date
0,15,F,Mumbai,CareOne,0,General,OPD,105,9.63,5,4,21.120000,1,3,Low,2026-01-01
1,3,M,Bangalore,CareOne,0,ICU,ICU,112,59.60,141,8,23.750000,1,3,High,2026-01-01
2,44,M,Pune,MediCareX,1,ICU,ER,150,59.28,165,3,32.460000,1,3,High,2026-01-01
3,47,F,Delhi,CareOne,1,General,OPD,145,25.15,60,3,30.056667,1,3,Medium,2026-01-01
4,14,F,Chennai,SecureLife,1,Cardiology,ER,148,42.88,330,9,29.030000,1,3,Medium,2026-01-01


In [8]:
# Time based split
risk_df = risk_df.sort_values("visit_date").reset_index(drop=True)
split_idx = int(len(risk_df)*0.8)

risk_train = risk_df.iloc[:split_idx].copy()
risk_test = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train[risk_features]
X_test_risk = risk_test[risk_features]

y_train_risk = risk_train[risk_target]
y_test_risk = risk_test[risk_target]

print("Train shape: ",X_train_risk.shape)
print("Test shape: ",X_test_risk.shape)
print("Train Period: ",risk_train["visit_date"].min().date(),"->", risk_train["visit_date"].max().date())
print("Test Period: ",risk_test["visit_date"].min().date(),"->", risk_test["visit_date"].max().date())

Train shape:  (20000, 14)
Test shape:  (5000, 14)
Train Period:  2025-01-21 -> 2026-01-02
Test Period:  2026-01-02 -> 2026-01-20


## Preprocessing Pipeline
**Two transformers:**
- Numeric -> SimpleImputer (median strategy)
- Categorical -> SimpleImputer -> OneHotEncoder


Combined using ColumnTransformer. this entire pipeline gets saved with the model -  no seperate preprocessing step at inference time.

In [9]:
# Preprocessing pipeline
risk_numeric_features = [
    "age",
    "chronic_flag",
    "length_of_stay_hours",
    "days_since_registration",
    "visit_frequency",
    "doctor_id",
    "avg_los_per_patient",
    "visit_month",
    "visit_dayofweek"
]

risk_categorical_features = [
    "gender",
    "city",
    "insurance_provider",
    "department",
    "visit_type"
]

risk_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

risk_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

risk_preprocessor = ColumnTransformer(transformers=[
    ("num", risk_numeric_transformer, risk_numeric_features),
    ("cat", risk_categorical_transformer, risk_categorical_features)
])

print("Preprocessing pipeline ready")

Preprocessing pipeline ready


## Baseline model: Logistic Regression

In [10]:
# Logistic Regression Model
risk_baseline_model = Pipeline(steps=[
    ("preprocessor", risk_preprocessor),
    ("classifier", LogisticRegression(
        max_iter = 1000,
        class_weight="balanced"
    ))
])

# Train
risk_baseline_model.fit(X_train_risk, y_train_risk)

# Predict
risk_baseline_pred_train = risk_baseline_model.predict(X_train_risk)
risk_baseline_pred_test =  risk_baseline_model.predict(X_test_risk)

baseline_acc = accuracy_score(y_test_risk, risk_baseline_pred_test)
print("Risk Baseline Train Accuracy :", round(accuracy_score(y_train_risk, risk_baseline_pred_train), 4))
print("Risk Baseline Test Accuracy  :", round(accuracy_score(y_test_risk,  risk_baseline_pred_test), 4))
print("Risk Baseline Test Weighted F1:", round(f1_score(y_test_risk, risk_baseline_pred_test, average="weighted"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test_risk, risk_baseline_pred_test))
print("Confusion Matrix:\n", confusion_matrix(y_test_risk, risk_baseline_pred_test))

c:\Users\ayata\anaconda33\envs\final_project\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Risk Baseline Train Accuracy : 0.9236
Risk Baseline Test Accuracy  : 0.9164
Risk Baseline Test Weighted F1: 0.9169

Classification Report:

              precision    recall  f1-score   support

        High       0.91      0.95      0.93       838
         Low       0.96      0.91      0.94      2360
      Medium       0.86      0.91      0.89      1802

    accuracy                           0.92      5000
   macro avg       0.91      0.92      0.92      5000
weighted avg       0.92      0.92      0.92      5000

Confusion Matrix:
 [[ 798    0   40]
 [   0 2142  218]
 [  82   78 1642]]


## Random Forest Model

In [11]:
# Random Forest Model
risk_rf_model = Pipeline(steps=[
    ("preprocessor", risk_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_split=20,
        min_samples_leaf=10,
        class_weight = "balanced_subsample",
        random_state=42
    ))
])

# Train
risk_rf_model.fit(X_train_risk, y_train_risk)

# Predict
risk_rf_pred_train = risk_rf_model.predict(X_train_risk)
risk_rf_pred_test =  risk_rf_model.predict(X_test_risk)

rf_acc = accuracy_score(y_test_risk, risk_rf_pred_test)
print("Risk Baseline Train Accuracy :", round(accuracy_score(y_train_risk, risk_rf_pred_train), 4))
print("Risk Baseline Test Accuracy  :", round(accuracy_score(y_test_risk,  risk_rf_pred_test), 4))
print("Risk Baseline Test Weighted F1:", round(f1_score(y_test_risk, risk_rf_pred_test, average="weighted"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test_risk, risk_rf_pred_test))
print("Confusion Matrix:\n", confusion_matrix(y_test_risk, risk_rf_pred_test))

Risk Baseline Train Accuracy : 0.9416
Risk Baseline Test Accuracy  : 0.933
Risk Baseline Test Weighted F1: 0.933

Classification Report:

              precision    recall  f1-score   support

        High       0.95      0.93      0.94       838
         Low       0.95      0.95      0.95      2360
      Medium       0.91      0.91      0.91      1802

    accuracy                           0.93      5000
   macro avg       0.93      0.93      0.93      5000
weighted avg       0.93      0.93      0.93      5000

Confusion Matrix:
 [[ 779    0   59]
 [   0 2249  111]
 [  45  120 1637]]


## XGBoost Model

In [12]:
# encode target for XGBoost only
le = LabelEncoder()
risk_df["risk_score_encoded"] = le.fit_transform(risk_df["risk_score"])
print("Label Mapping: ",dict(zip(le.classes_,le.transform(le.classes_))))

Label Mapping:  {'High': 0, 'Low': 1, 'Medium': 2}


In [13]:
# Time-based split for XGBoost
# Using same split_idx — consistent with LR and RF
risk_df = risk_df.sort_values("visit_date").reset_index(drop=True)
split_idx = int(len(risk_df) * 0.8)

risk_train_xgb = risk_df.iloc[:split_idx].copy()
risk_test_xgb  = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train_xgb[risk_features]
X_test_risk  = risk_test_xgb[risk_features]

# Separate XGB label variables — does NOT overwrite y_train_risk
y_train_risk_xgb = risk_train_xgb["risk_score_encoded"]
y_test_risk_xgb  = risk_test_xgb["risk_score_encoded"]

print("XGBoost train shape:", X_train_risk.shape)
print("XGBoost test shape :", X_test_risk.shape)

XGBoost train shape: (20000, 14)
XGBoost test shape : (5000, 14)


In [14]:
# XGBoost Model
risk_xgb_model = Pipeline(steps=[
    ("preprocessor", risk_preprocessor),
    ("classifier", XGBClassifier(
        objective="multi:softmax",
        num_class=3,
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    ))
])

# Train
risk_xgb_model.fit(X_train_risk, y_train_risk_xgb)

# Predict Train
risk_xgb_pred_train = risk_xgb_model.predict(X_train_risk)
# Predict Test
risk_xgb_pred_test  = risk_xgb_model.predict(X_test_risk)

# Save for comparison
xgb_acc = accuracy_score(y_test_risk_xgb, risk_xgb_pred_test)

print("Risk XGB Train Accuracy :", round(accuracy_score(y_train_risk_xgb, risk_xgb_pred_train), 4))
print("Risk XGB Test Accuracy  :", round(accuracy_score(y_test_risk_xgb,  risk_xgb_pred_test),  4))
print("Risk XGB Test Weighted F1:", round(f1_score(y_test_risk_xgb, risk_xgb_pred_test, average="weighted"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test_risk_xgb, risk_xgb_pred_test,
                             target_names=le.classes_))
print("Confusion Matrix:\n", confusion_matrix(y_test_risk_xgb, risk_xgb_pred_test))

Risk XGB Train Accuracy : 0.9945
Risk XGB Test Accuracy  : 0.9568
Risk XGB Test Weighted F1: 0.9569

Classification Report:

              precision    recall  f1-score   support

        High       0.98      0.94      0.96       840
         Low       0.97      0.97      0.97      2360
      Medium       0.93      0.95      0.94      1800

    accuracy                           0.96      5000
   macro avg       0.96      0.95      0.96      5000
weighted avg       0.96      0.96      0.96      5000

Confusion Matrix:
 [[ 789    0   51]
 [   0 2284   76]
 [  15   74 1711]]


In [15]:
# Model Comparison
print("="*45)
print("Model A - RISK CLASSIFICATION COMPARISON")
print("="*45)
print(f"Logistic Regression: {baseline_acc:.4f}")
print(f"Random Forest      : {rf_acc:.4f}")
print(f"XGBoost            : {xgb_acc:.4f}")
print()
print(f"Best model Xgboost -> {max(baseline_acc, rf_acc, xgb_acc)}")

Model A - RISK CLASSIFICATION COMPARISON
Logistic Regression: 0.9164
Random Forest      : 0.9330
XGBoost            : 0.9568

Best model Xgboost -> 0.9568


# Model B -- Claim Outcome Prediction
Buisness Purpose: Predict which insurance claim will be Paid,Pending or Rejected.

Target: claim_status

Split: Time-based on billing_date

Leakage prevention:
approved_amount and payment_days excluded

In [19]:
# Defining features
claim_target = 'claim_status'

claim_features = [
    "age",
    "gender",
    "city",
    "insurance_provider",
    "chronic_flag",
    "department",
    "visit_type",
    "doctor_id",
    "length_of_stay_hours",
    "risk_score",
    "billed_amount",
    "days_since_registration",
    "visit_frequency",
    "avg_los_per_patient",
    "provider_rejection_rate",
    "visit_month",
    "visit_dayofweek",
    "high_cost_visit_flag"
]

claim_df = df[claim_features + [claim_target, "billing_date"]].copy()
claim_df = claim_df.dropna(subset=[claim_target,"billing_date"])
print("Claim Dataset Shape: ",claim_df.shape)
claim_df.head()

Claim Dataset Shape:  (25000, 20)


,age,gender,city,insurance_provider,chronic_flag,department,visit_type,doctor_id,length_of_stay_hours,risk_score,billed_amount,days_since_registration,visit_frequency,avg_los_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag,claim_status,billing_date
0,15,F,Mumbai,CareOne,0,General,OPD,105,9.63,Low,9612.77,5,4,21.120000,0.256876,1,3,False,Rejected,2026-01-19
1,3,M,Bangalore,CareOne,0,ICU,ICU,112,59.60,High,88539.01,141,8,23.750000,0.256876,1,3,True,Rejected,2026-01-05
2,44,M,Pune,MediCareX,1,ICU,ER,150,59.28,High,88539.01,165,3,32.460000,0.242556,1,3,True,Pending,2026-01-20
3,47,F,Delhi,CareOne,1,General,OPD,145,25.15,Medium,20958.52,60,3,30.056667,0.256876,1,3,False,Paid,2026-01-15
4,14,F,Chennai,SecureLife,1,Cardiology,ER,148,42.88,Medium,74921.48,330,9,29.030000,0.157496,1,3,True,Paid,2026-01-16


In [20]:
# claim status distribution
print("Claim Status Distribution")
print(claim_df[claim_target].value_counts())
print()
print("Claim Status % Distribution")
print((claim_df[claim_target].value_counts(normalize=True)*100).round(2))

Claim Status Distribution
claim_status
Paid        13638
Pending      6096
Rejected     5266
Name: count, dtype: int64

Claim Status % Distribution
claim_status
Paid        54.55
Pending     24.38
Rejected    21.06
Name: proportion, dtype: float64


## Time-Based Split for Claim Model

Sorting by billing_date - not visit_date. Billing events after visits

In [22]:
claim_df = claim_df.sort_values("billing_date").reset_index(drop=True)
split_idx = int(len(claim_df)*0.8)

claim_train = claim_df.iloc[:split_idx].copy()
claim_test = claim_df.iloc[split_idx:].copy()

X_train_claim = claim_train[claim_features]
X_test_claim = claim_test[claim_features]

y_train_claim = claim_train[claim_target]
y_test_claim = claim_test[claim_target]

print("Train shape:", X_train_claim.shape)
print("Test shape:", X_test_claim.shape)
print("Train period:", claim_train["billing_date"].min().date(),
      "=>",claim_train["billing_date"].max().date())
print("Test period:", claim_test["billing_date"].min().date(),
      "=>",claim_test["billing_date"].max().date())

Train shape: (20000, 18)
Test shape: (5000, 18)
Train period: 2025-01-28 => 2026-01-17
Test period: 2026-01-17 => 2026-01-20


## Preprocessing Pipeline for Claim Model

**Similar to risk model but claim model had additional features:**
- risk_score as categorical input feature
- billed_amount as numeric
- provider_rejection_rate as numeric

In [29]:
claim_numeric_features = [
    "age", "chronic_flag","doctor_id",
    "length_of_stay_hours","billed_amount",
    "days_since_registration","visit_frequency",
    "avg_los_per_patient","provider_rejection_rate",
    "visit_month","visit_dayofweek","high_cost_visit_flag"
]

claim_categorical_features = [
    "gender","city","insurance_provider",
    "department","visit_type","risk_score"
]

claim_numeric_transformer = Pipeline(steps=[
    ("impute",SimpleImputer(strategy="median"))
])

claim_categorical_transformer = Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("onehot",OneHotEncoder(handle_unknown="ignore"))
])

claim_preprocessor = ColumnTransformer(transformers=[
    ("num",claim_numeric_transformer, claim_numeric_features),
    ("cat", claim_categorical_transformer, claim_categorical_features)
])

print("Claim Preprocessing pipeline ready")

Claim Preprocessing pipeline ready


In [30]:
# Logistic regression model
claim_baseline_model = Pipeline(steps=[
    ("preprocessor",claim_preprocessor),
    ("classifier",LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

claim_baseline_model.fit(X_train_claim,y_train_claim)
claim_baseline_pred_train = claim_baseline_model.predict(X_train_claim)
claim_baseline_pred_test = claim_baseline_model.predict(X_test_claim)
# evaluate
claim_baseline_acc = accuracy_score(y_test_claim,claim_baseline_pred_test)

print("Claim Baseline Train Accuracy: ",round(accuracy_score(y_train_claim,claim_baseline_pred_train),4))
print("Claim Baseline Test Accuracy: ",round(accuracy_score(y_test_claim,claim_baseline_pred_test),4))
print("Claim Baseline Test Weighted F1: ",round(f1_score(y_test_claim,claim_baseline_pred_test,average="weighted"),4))
print("\nClassification Report:")
print(classification_report(y_test_claim,claim_baseline_pred_test))
print("Confusion matrix:\n",confusion_matrix(y_test_claim,claim_baseline_pred_test))


c:\Users\ayata\anaconda33\envs\final_project\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Claim Baseline Train Accuracy:  0.4849
Claim Baseline Test Accuracy:  0.414
Claim Baseline Test Weighted F1:  0.4267

Classification Report:
              precision    recall  f1-score   support

        Paid       0.64      0.50      0.56      2773
     Pending       0.24      0.20      0.22      1188
    Rejected       0.25      0.44      0.32      1039

    accuracy                           0.41      5000
   macro avg       0.37      0.38      0.36      5000
weighted avg       0.46      0.41      0.43      5000

Confusion matrix:
 [[1376  527  870]
 [ 440  236  512]
 [ 346  235  458]]


In [44]:
# Logistic regression model
claim_rf_model = Pipeline(steps=[
    ("preprocessor",claim_preprocessor),
    ("classifier",RandomForestClassifier(
        n_estimators=250,
        max_depth=14,
        min_samples_split=8,
        class_weight = "balanced",
        random_state=42
    ))
])

claim_rf_model.fit(X_train_claim,y_train_claim)
claim_rf_pred_train = claim_rf_model.predict(X_train_claim)
claim_rf_pred_test = claim_rf_model.predict(X_test_claim)
# evaluate
claim_rf_acc = accuracy_score(y_test_claim,claim_rf_pred_test)

print("Claim Random Forest Train Accuracy: ",round(accuracy_score(y_train_claim,claim_rf_pred_train),4))
print("Claim Random Forest Test Accuracy: ",round(accuracy_score(y_test_claim,claim_rf_pred_test),4))
print("Claim Random Forest Test Weighted F1: ",round(f1_score(y_test_claim,claim_rf_pred_test,average="weighted"),4))
print("\nClassification Report:")
print(classification_report(y_test_claim,claim_rf_pred_test))
print("Confusion matrix:\n",confusion_matrix(y_test_claim,claim_rf_pred_test))


Claim Random Forest Train Accuracy:  0.8554
Claim Random Forest Test Accuracy:  0.5316
Claim Random Forest Test Weighted F1:  0.4988

Classification Report:
              precision    recall  f1-score   support

           0       0.62      0.79      0.69      2773
           1       0.32      0.20      0.24      1188
           2       0.31      0.23      0.27      1039

    accuracy                           0.53      5000
   macro avg       0.42      0.41      0.40      5000
weighted avg       0.49      0.53      0.50      5000

Confusion matrix:
 [[2185  291  297]
 [ 726  233  229]
 [ 604  195  240]]


In [34]:
claim_df

,age,gender,city,insurance_provider,chronic_flag,department,visit_type,doctor_id,length_of_stay_hours,risk_score,billed_amount,days_since_registration,visit_frequency,avg_los_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag,claim_status,billing_date
0,75,M,Chennai,HealthPlus,1,ICU,ICU,178,68.76,High,88539.01,3,5,39.466000,0.186952,1,4,True,Pending,2025-01-28
1,45,F,Pune,SecureLife,0,ICU,ER,160,57.60,Medium,88539.01,2,10,26.282000,0.157496,1,6,True,Paid,2025-01-31
2,20,M,Chennai,MediCareX,0,ER,ER,107,8.81,Low,12441.60,3,7,14.450000,0.242556,1,1,False,Rejected,2025-02-01
3,83,M,Chennai,SecureLife,1,General,OPD,196,29.20,Medium,29935.51,11,5,33.024000,0.157496,1,4,False,Paid,2025-02-01
4,67,M,Delhi,MediCareX,1,General,OPD,110,14.96,Low,11467.38,10,7,19.325714,0.242556,1,3,False,Paid,2025-02-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,28,M,Delhi,CareOne,0,General,OPD,176,6.17,Low,6307.15,229,6,13.388333,0.256876,12,2,False,Rejected,2026-01-20
24996,61,M,Bangalore,SecureLife,1,ICU,ICU,184,62.63,High,88539.01,238,6,25.833333,0.157496,1,3,True,Paid,2026-01-20
24997,44,F,Chennai,HealthPlus,0,ER,ER,171,3.96,Low,4915.19,5,2,13.655000,0.186952,1,3,False,Paid,2026-01-20
24998,26,M,Mumbai,CareOne,0,ICU,ICU,148,5.96,Medium,19048.24,47,3,23.193333,0.256876,1,5,False,Paid,2026-01-20


In [35]:
# encode target for XGBoost only
le = LabelEncoder()
claim_df["claim_status_encoded"] = le.fit_transform(claim_df["claim_status"])
print("Label Mapping: ",dict(zip(le.classes_,le.transform(le.classes_))))

Label Mapping:  {'Paid': 0, 'Pending': 1, 'Rejected': 2}


In [41]:
y_train_claim = le.fit_transform(y_train_claim)
y_test_claim = le.fit_transform(y_test_claim)

In [43]:
# Logistic regression model
claim_xgb_model = Pipeline(steps=[
    ("preprocessor",claim_preprocessor),
    ("classifier",XGBClassifier(
        objective="multi:softmax",
        num_class=3,
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    ))
])

claim_xgb_model.fit(X_train_claim,y_train_claim)
claim_xgb_pred_train = claim_xgb_model.predict(X_train_claim)
claim_xgb_pred_test = claim_xgb_model.predict(X_test_claim)
# evaluate
claim_xgb_acc = accuracy_score(y_test_claim,claim_xgb_pred_test)

print("Claim Random Forest Train Accuracy: ",round(accuracy_score(y_train_claim,claim_xgb_pred_train),4))
print("Claim Random Forest Test Accuracy: ",round(accuracy_score(y_test_claim,claim_xgb_pred_test),4))
print("Claim Random Forest Test Weighted F1: ",round(f1_score(y_test_claim,claim_xgb_pred_test,average="weighted"),4))
print("\nClassification Report:")
print(classification_report(y_test_claim,claim_xgb_pred_test))
print("Confusion matrix:\n",confusion_matrix(y_test_claim,claim_xgb_pred_test))


Claim Random Forest Train Accuracy:  0.7659
Claim Random Forest Test Accuracy:  0.5454
Claim Random Forest Test Weighted F1:  0.4707

Classification Report:
              precision    recall  f1-score   support

           0       0.59      0.89      0.71      2773
           1       0.31      0.10      0.15      1188
           2       0.33      0.14      0.20      1039

    accuracy                           0.55      5000
   macro avg       0.41      0.38      0.35      5000
weighted avg       0.47      0.55      0.47      5000

Confusion matrix:
 [[2459  152  162]
 [ 926  120  142]
 [ 780  111  148]]


In [45]:
# Model Comparison
print("="*45)
print("Model B - CLAIM OUTCOME COMPARISON")
print("="*45)
print(f"Logistic Regression: {claim_baseline_acc:.4f}")
print(f"Random Forest      : {claim_rf_acc:.4f}")
print(f"XGBoost            : {claim_xgb_acc:.4f}")
print()
print(f"Best model Xgboost -> {max(claim_baseline_acc, claim_rf_acc, claim_xgb_acc)}")

Model B - CLAIM OUTCOME COMPARISON
Logistic Regression: 0.4140
Random Forest      : 0.5316
XGBoost            : 0.5454

Best model Xgboost -> 0.5454


# Saving Model Artifacts

In [46]:
joblib.dump(risk_xgb_model, "../models/risk_model.joblib")
joblib.dump(claim_xgb_model, "../models/claim_model.joblib")

print("Models saved")

Models saved


## Saving Feature schema
**The feature schema will tell our fastAPI:**
- which features to expect in the request payload
- which column is the target
- what split strategy was used

In [47]:
feature_schema = {
    "risk_model_features" : risk_features, # list of 14 features
    "claim_model_features": claim_features, # list of 18 features
    "risk_target": "risk_score", # what model A predicts
    "claim_target" : "claim_status", # what model B predicts
    "risk_time_column" : "visit_date", # split column for model A
    "claim_time_column": "billing_date", # split column foe model B
    "split_strategy":"earliest 80 percent train, latest 20 percent test"
}

with open("../outputs/feature_schema.json","w") as f:
    json.dump(feature_schema,f,indent=4)

print("feature schema saved")

feature schema saved
